In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

# Regression: Predicting Rental Price
In this notebook, we will use the dataset we cleansed in the previous lab to predict Airbnb rental prices

## Load Dataset
Let's load the clean Airbnb dataset in again 
We created it in the previous notebook, it should exists in `/home/jovyan/work/datasets/output/airbnb/clean_data`

In [2]:
file_path = f"/home/jovyan/work/datasets/output/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)

In [4]:
train_df, test_df = airbnb_df.randomSplit([.8, .2], seed=42)
print(train_df.cache().count())
print(test_df.cache().count())

14283
3502


In [5]:
from pyspark.sql.functions import avg, lit, median
from pyspark.ml.evaluation import RegressionEvaluator

avg_price = train_df.select(avg("price")).first()[0]
median_price = train_df.select(median("price")).first()[0]

pred_df = (test_df
          .withColumn("avgPrediction", lit(avg_price))
          .withColumn("medianPrediction", lit(median_price)))

regression_evaluatorAVG = RegressionEvaluator(predictionCol="avgPrediction", labelCol="price", metricName="rmse")
regression_evaluatorMedian = RegressionEvaluator(predictionCol="medianPrediction", labelCol="price", metricName="rmse")


rmseAVG = regression_evaluatorAVG.evaluate(pred_df)
print(f"RMSE for AVG is: {rmseAVG}")
rmseMedian = regression_evaluatorMedian.evaluate(pred_df)
print(f"RMSE for MEDIAN is {rmseMedian}")


r2Avg = regression_evaluatorAVG.setMetricName("r2").evaluate(pred_df)
print(f"R2 for AVG is {r2Avg}")
r2Median = regression_evaluatorMedian.setMetricName("r2").evaluate(pred_df)
print(f"R2 for Median is {r2Median}")

RMSE for AVG is: 57.72840841939142
RMSE for MEDIAN is 58.37921776470596
R2 for AVG is -5.960639805380197e-05
R2 for Median is -0.022735334682391084


## Linear Regression
Check **`price`** and **`bedrooms`** relations with a visualization

In [7]:
train_df.select("price", "bedrooms").show()

+-----+--------+
|price|bedrooms|
+-----+--------+
| 79.0|     1.0|
|157.0|     1.0|
| 85.0|     1.0|
|122.0|     1.0|
|143.0|     1.0|
| 85.0|     1.0|
| 44.0|     1.0|
| 62.0|     1.0|
| 49.0|     1.0|
|225.0|     2.0|
| 96.0|     1.0|
|126.0|     1.0|
|122.0|     1.0|
|115.0|     3.0|
|150.0|     3.0|
|112.0|     1.0|
| 70.0|     1.0|
| 85.0|     1.0|
|200.0|     1.0|
| 65.0|     1.0|
+-----+--------+
only showing top 20 rows



In [8]:
display(train_df.select("price", "bedrooms").summary().toPandas())

,summary,price,bedrooms
0,count,14283,14283
1,mean,118.25848911293146,1.2255128474410137
2,stddev,57.4655182729535,0.8297364753753466
3,min,8.0,0.0
4,25%,70.0,1.0
5,50%,110.0,1.0
6,75%,160.0,1.0
7,max,249.0,7.0


Our dataset has a lot of columns, we'll be using only two of them for this notebook for the sake of simplicity
* bedrooms: Feature
* price: Label

We will use [LinearRegression](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.LinearRegression.html?highlight=linearregression#pyspark.ml.regression.LinearRegression) to build the model.
We will also use [VectorAssembler](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html?highlight=vectorassembler#pyspark.ml.feature.VectorAssembler) to build the feature column to the proper type

In [9]:
#Sample Vector Assembler

from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler

dataset = spark.createDataFrame(
    [(0, 18, 1.0, 1.0),(1, 22, 3.0, 5.0)],
    ["id", "hour", "mobile", "clicked"])

assembler = VectorAssembler(
    inputCols=["hour", "mobile"],
    outputCol="features")

print("original dataset")
dataset.show(truncate=False)

output = assembler.transform(dataset)

print("result dataset")
output.select("id", "features", "clicked").show(truncate=False)

original dataset
+---+----+------+-------+
|id |hour|mobile|clicked|
+---+----+------+-------+
|0  |18  |1.0   |1.0    |
|1  |22  |3.0   |5.0    |
+---+----+------+-------+

result dataset
+---+----------+-------+
|id |features  |clicked|
+---+----------+-------+
|0  |[18.0,1.0]|1.0    |
|1  |[22.0,3.0]|5.0    |
+---+----------+-------+



In [10]:
from pyspark.ml.feature import VectorAssembler

vec_assembler = VectorAssembler(inputCols=["bedrooms"], outputCol="features")

vec_train_df = vec_assembler.transform(train_df)

In [11]:
vec_train_df.printSchema()

root
 |-- host_is_superhost: string (nullable = true)
 |-- instant_bookable: string (nullable = true)
 |-- host_total_listings_count: double (nullable = true)
 |-- neighbourhood_cleansed: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- accommodates: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- beds: double (nullable = true)
 |-- minimum_nights: double (nullable = true)
 |-- number_of_reviews: double (nullable = true)
 |-- review_scores_rating: double (nullable = true)
 |-- review_scores_accuracy: double (nullable = true)
 |-- review_scores_cleanliness: double (nullable = true)
 |-- review_scores_checkin: double (nullable = true)
 |-- review_scores_communication: double (nullable = true)
 |-- review_scores_location: double (nullable = true)
 |-- review_scores_value: double (n

In [12]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="price")
lr_model = lr.fit(vec_train_df)

In [13]:
print(lr.explainParams())

aggregationDepth: suggested depth for treeAggregate (>= 2). (default: 2)
elasticNetParam: the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty. (default: 0.0)
epsilon: The shape parameter to control the amount of robustness. Must be > 1.0. Only valid when loss is huber (default: 1.35)
featuresCol: features column name. (default: features, current: features)
fitIntercept: whether to fit an intercept term. (default: True)
labelCol: label column name. (default: label, current: price)
loss: The loss function to be optimized. Supported options: squaredError, huber. (default: squaredError)
maxBlockSizeInMB: maximum memory in MB for stacking input data into blocks. Data is stacked within partitions. If more than remaining data size in a partition then it is adjusted to the data size. Default 0.0 represents choosing optimal value, depends on specific algorithm. Must be >= 0. (default: 0.0)
maxIter: max number of itera

## Inspect the Model
We can extract the formula for the lineal regression where:
* Formula: y = Coefficient * X + Intercept

In [14]:
m = lr_model.coefficients[0]
b = lr_model.intercept

print(f"The formula for the linear regression line is y = {m:.2f}x + {b:.2f}")

The formula for the linear regression line is y = 2.99x + 114.59


## Apply Model to Test Set
* First transform the test_df with Vector Assembler as we did with train_df
* Instead of fit method, which is used to training, we use transform method from the model, it will create a column named prediction

In [16]:
vec_test_df = vec_assembler.transform(test_df)

pred_df = lr_model.transform(vec_test_df)

In [19]:
display(pred_df.select("bedrooms", "price", "prediction").show())

+--------+-----+------------------+
|bedrooms|price|        prediction|
+--------+-----+------------------+
|     1.0|130.0|117.58356664637509|
|     1.0| 65.0|117.58356664637509|
|     1.0| 43.0|117.58356664637509|
|     1.0| 52.0|117.58356664637509|
|     2.0|106.0|120.57640042154567|
|     1.0|105.0|117.58356664637509|
|     2.0|160.0|120.57640042154567|
|     1.0| 95.0|117.58356664637509|
|     1.0| 67.0|117.58356664637509|
|     1.0| 33.0|117.58356664637509|
|     1.0| 39.0|117.58356664637509|
|     1.0| 65.0|117.58356664637509|
|     1.0|121.0|117.58356664637509|
|     1.0| 87.0|117.58356664637509|
|     1.0| 99.0|117.58356664637509|
|     1.0| 48.0|117.58356664637509|
|     5.0| 75.0|129.55490174705741|
|     1.0|245.0|117.58356664637509|
|     3.0|185.0|123.56923419671625|
|     1.0|100.0|117.58356664637509|
+--------+-----+------------------+
only showing top 20 rows



None

## Evaluate the Model

In [20]:
from pyspark.ml.evaluation import RegressionEvaluator

regression_evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="price", metricName="rmse")

rmse = regression_evaluator.evaluate(pred_df)
print(f"RMSE is {rmse}")
r2 = regression_evaluator.setMetricName("r2").evaluate(pred_df)
print(f"R2 is {r2}")

RMSE is 57.677417969267964
R2 is 0.0017062821849358478
